In [ ]:
import pandas as pd
import numpy as np
import os

# Load the dataset
input_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/B-Featuring Removing/2features24.csv'
df = pd.read_csv(input_path)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print("\n" + "="*60)

# List of the first 11 variables for analysis (including h05_chupeta_usou)
variables_to_analyze = [
    'id_anon',
    'a00_regiao', 
    'b02_sexo',
    'b04_idade',
    'bb04_idade_da_mae',
    'd01_cor',
    'h01_semanas_gravidez',
    'h02_peso',
    'h03_altura',
    'h04_parto',
    'h05_chupeta_usou'
]

def analyze_variable(df, var_name, max_unique=10):
    """
    Analyzes a specific variable from the dataset
    
    Args:
        df: DataFrame
        var_name: variable name
        max_unique: maximum number of unique values to display
    """
    print(f"\n[ANALYSIS] VARIABLE ANALYSIS: {var_name}")
    print("-" * 50)
    
    # Basic information
    print(f"Data type: {df[var_name].dtype}")
    print(f"Unique values: {df[var_name].nunique()}")
    print(f"Missing values: {df[var_name].isna().sum()} ({df[var_name].isna().mean()*100:.1f}%)")
    
    # If numeric, show descriptive statistics
    if df[var_name].dtype in ['int64', 'float64']:
        print("\nDescriptive statistics:")
        print(df[var_name].describe())
        
        # If it has few unique values, show value counts
        if df[var_name].nunique() <= max_unique:
            print(f"\nValue distribution:")
            print(df[var_name].value_counts().sort_index())
    
    # If categorical or has few unique values
    elif df[var_name].nunique() <= max_unique:
        print(f"\nValue distribution:")
        counts = df[var_name].value_counts()
        for value, count in counts.items():
            percentage = (count / len(df)) * 100
            print(f"  {value}: {count} ({percentage:.1f}%)")
    
    # If it has many unique values, show only the most frequent
    else:
        print(f"\nTop {max_unique} most frequent values:")
        counts = df[var_name].value_counts().head(max_unique)
        for value, count in counts.items():
            percentage = (count / len(df)) * 100
            print(f"  {value}: {count} ({percentage:.1f}%)")

# Analyze each variable
for var in variables_to_analyze:
    if var in df.columns:
        analyze_variable(df, var)
    else:
        print(f"\n[WARNING] VARIABLE {var} NOT FOUND IN DATASET")

print("\n" + "="*60)
print("[SEARCH] EXPLORATORY ANALYSIS COMPLETED")
print("="*60)

# Check if the output directory exists
output_dir = '/Users/marcelosilva/Desktop/early-obesity-prediction/C-Feature Engeneering'
if not os.path.exists(output_dir):
    print(f"\n[DIR] Creating directory: {output_dir}")
    os.makedirs(output_dir, exist_ok=True)

print(f"\n[INFO] NEXT STEPS:")
print("1. Review the variable distributions above")
print("2. Confirm the transformations to be applied")
print("3. Run the feature engineering script")
print(f"4. Save result to: {output_dir}/1FE1-10.csv")

# Feature Engineering Report - First 11 Variables

## Project Overview

This report documents the feature engineering process applied to the first 11 variables from a dataset of 8,236 Brazilian children aged 2-4 years, focusing on nutritional outcome prediction based on the first 24 months of life.

## Methodology

### Sequential Approach
- Variable-by-variable processing for better control
- Exploratory analysis prior to transformations
- Domain knowledge-based transformations
- Complete dataset preservation during process

### Transformation Strategies
1. **Categorical variables**: One-hot encoding with intelligent grouping
2. **Binary variables**: Semantically clear 0/1 encoding
3. **Feature engineering**: Creation of clinically relevant derived variables
4. **Missing value treatment**: Conservative median imputation

## Implemented Transformations

### 1. Identification and Demographics

#### `id_anon` → `id_anon`
- **Status**: Kept unchanged
- **Function**: Unique identifier for record tracking
- **Values**: 8,236 unique IDs (no duplicates)

#### `a00_regiao` → 5 dummy variables
- **Transformation**: Complete one-hot encoding
- **Result**: `regiao_Centro-Oeste`, `regiao_Nordeste`, `regiao_Norte`, `regiao_Sudeste`, `regiao_Sul`
- **Distribution**: Relatively balanced (18.8% - 21.6%)
- **Rationale**: Important variable for regional equity analyses

#### `b02_sexo` → `sexo_masculino`
- **Transformation**: Binary encoding
- **Encoding**: 0 = Female (48.8%), 1 = Male (51.2%)
- **Result**: Balanced distribution between sexes

#### `d01_cor` → 4 dummy variables (grouped)
- **Transformation**: Grouping + one-hot encoding
- **Grouping performed**: "Yellow" (0.5%) + "Indigenous" (0.2%) → "Others" (0.7%)
- **Result**: `cor_Branca` (39.2%), `cor_Parda` (53.3%), `cor_Preta` (6.9%), `cor_Outros` (0.7%)
- **Rationale**: Avoid overfitting with very rare categories

### 2. Age and Timing Variables

#### `b04_idade` → `idade_crianca`
- **Status**: Kept with renaming
- **Distribution**: 2 years (33.0%), 3 years (34.1%), 4 years (32.9%)
- **Note**: Balanced distribution across ages

#### `bb04_idade_da_mae` → `idade_materna_ao_nascimento`
- **Transformation**: Feature engineering (bb04 - b04)
- **Missing values**: 7 cases (0.1%) imputed with median (29.0 years)
- **Rationale**: Maternal age at birth more relevant than current age for first 24 months outcomes
- **Resulting range**: ~12-78 years (maternal age at birth)

### 3. Perinatal Variables

#### `h01_semanas_gravidez` → `semanas_gravidez`
- **Status**: Kept with renaming
- **Statistics**: Mean 38.8 weeks (SD: 2.2)
- **Range**: 26-43 weeks
- **Prematurity**: ~15% born <37 weeks

#### `h02_peso` → `peso_nascimento`
- **Status**: Kept with renaming  
- **Statistics**: Mean 3,188g (SD: 595g)
- **Range**: 250-5,900g
- **Low birth weight**: ~10% <2,500g
- **Note**: Extreme values require further analysis

#### `h03_altura` → `altura_nascimento`
- **Status**: Kept with renaming
- **Statistics**: Mean 48.5cm (SD: 3.3cm)
- **Range**: 27-61cm

#### `h04_parto` → Matrix + binary variable
- **Dual transformation**:
  1. **Matrix**: `parto_Normal` (53.2%), `parto_Cesariana_urgencia` (25.6%), `parto_Cesariana_agendada` (21.2%)
  2. **Binary**: `tipo_de_parto` (0=Normal, 1=Cesarean total)
- **Cesarean rate**: 46.8% (high by WHO standards)

### 4. Child Care Practices

#### `h05_chupeta_usou` → `chupeta_usou`
- **Transformation**: Logical grouping + binary encoding
- **Grouping**:
  - **"Used" (1)**: "Previously used" + "Currently uses" = 41.2%
  - **"Never used" (0)**: "Refused" + "Never offered" + "Don't know" = 58.8%
- **Rationale**: Focus on usage experience vs non-usage

## Transformation Impact

### Dataset Expansion
- **Original variables processed**: 11
- **Variables resulting from transformations**: 21
- **Variables kept unchanged**: 45
- **Total in final dataset**: 66 variables

### Data Quality
- **Missing values**: 0 (all treated)
- **Imputation method**: Median for numerical variables
- **Imputed cases**: 7 (0.1% of dataset)

### Resulting Data Types
- **Binary (0/1)**: 17 variables (dummies + binary transformations)
- **Continuous numerical**: 4 variables (ages, anthropometric measures)
- **Identifiers**: 1 variable (id_anon)

## Clinical and Methodological Considerations

### Strengths
1. **Information preservation**: All clinically relevant categories maintained
2. **Intelligent handling of rare categories**: Grouping based on frequency and relevance
3. **Informed feature engineering**: Creation of variables more relevant to study period
4. **Reproducibility**: Documented and coded process

### Limitations and Next Steps
1. **Outliers**: Extreme values in weight (250g, 5,900g) and height (27cm, 61cm) require investigation
2. **Validation**: Correlation analysis needed between created variables
3. **Imbalance**: High cesarean rate (46.8%) may impact models
4. **Continuation**: 45 remaining variables await processing

### Machine Learning Implications
- **Multicollinearity**: Dummy variables may generate perfect correlation
- **Interpretability**: Transformations maintain clinical meaning
- **Scalability**: Standardized process for applying to remaining variables

## Conclusion

Feature engineering of the first 11 variables was successfully executed, resulting in transformations that preserve clinically relevant information while preparing data for machine learning analyses. The established methodological approach can be replicated on remaining variables, maintaining consistency and quality in the data preparation process.

The resulting dataset maintains original data integrity while expanding analytical possibilities through clinically informed engineered variables.

In [4]:
import pandas as pd
import numpy as np
import os

def feature_engineering_1_11():
    """
    Feature Engineering of the first 11 variables do dataset
    """
    
    # Load dataset
    input_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/B-Featuring Removing/2features24.csv'
    df = pd.read_csv(input_path)
    
    print(f"Original dataset: {df.shape}")
    print("Starting Feature Engineering of the first 11 variables...")
    print("=" * 60)
    
    # Create new dataframe with transformations
    df_fe = pd.DataFrame()
    
    # 1. id_anon - manter igual
    df_fe['id_anon'] = df['id_anon']
    print("[OK] id_anon: kept as identifier")
    
    # 2. a00_regiao - one-hot encoding (matriz)
    regiao_dummies = pd.get_dummies(df['a00_regiao'], prefix='regiao', drop_first=False)
    df_fe = pd.concat([df_fe, regiao_dummies], axis=1)
    print(f"[OK] a00_regiao: converted to {regiao_dummies.shape[1]} dummy variables")
    print(f"   Regions: {list(regiao_dummies.columns)}")
    
    # 3. b02_sexo - transformar em binário (0 = Feminino, 1 = Masculino)
    df_fe['sexo_masculino'] = (df['b02_sexo'] == 'Masculino').astype(int)
    print("[OK] b02_sexo: converted to sexo_masculino (0=Feminino, 1=Masculino)")
    
    # 4. b04_idade - manter
    df_fe['idade_crianca'] = df['b04_idade']
    print("[OK] b04_idade: kept as idade_crianca")
    
    # 5. bb04_idade_da_mae - criar nova variable: idade materna ao nascimento
    # Tratar missing values primeiro (usar mediana)
    idade_mae_median = df['bb04_idade_da_mae'].median()
    df_temp_mae = df['bb04_idade_da_mae'].fillna(idade_mae_median)
    df_fe['idade_materna_ao_nascimento'] = df_temp_mae - df['b04_idade']
    print(f"[OK] bb04_idade_da_mae: created idade_materna_ao_nascimento (bb04 - b04)")
    print(f"   Missing values filled with median: {idade_mae_median}")
    
    # 6. d01_cor - agrupar categorias pequenas em "Outros" e criar matriz
    # Amarela (0.5%) + Indígena (0.2%) = Outros (0.7%)
    df_cor = df['d01_cor'].copy()
    df_cor = df_cor.replace({
        'Amarela (origem japonesa, chinesa, coreana etc.)': 'Outros',
        'Indígena': 'Outros'
    })
    cor_dummies = pd.get_dummies(df_cor, prefix='cor', drop_first=False)
    df_fe = pd.concat([df_fe, cor_dummies], axis=1)
    print("[OK] d01_cor: small categories grouped into 'Outros' e converted to matriz")
    print(f"   Categories: {list(cor_dummies.columns)}")
    
    # 7. h01_semanas_gravidez - manter
    df_fe['semanas_gravidez'] = df['h01_semanas_gravidez']
    print("[OK] h01_semanas_gravidez: kept as semanas_gravidez")
    
    # 8. h02_peso - manter
    df_fe['peso_nascimento'] = df['h02_peso']
    print("[OK] h02_peso: kept as peso_nascimento")
    
    # 9. h03_altura - manter
    df_fe['altura_nascimento'] = df['h03_altura']
    print("[OK] h03_altura: kept as altura_nascimento")
    
    # 10. h04_parto - criar matriz + variable binária tipo_de_parto
    # Matriz com todas as categorias
    parto_dummies = pd.get_dummies(df['h04_parto'], prefix='parto', drop_first=False)
    df_fe = pd.concat([df_fe, parto_dummies], axis=1)
    
    # Variable binária: 0 = Normal, 1 = Cesariana (urgência ou agendada)
    df_fe['tipo_de_parto'] = (~df['h04_parto'].str.contains('Normal', na=False)).astype(int)
    
    print("[OK] h04_parto: converted to matrix + binary variable tipo_de_parto")
    print(f"   Matrix categories: {list(parto_dummies.columns)}")
    print("   tipo_de_parto: 0=Normal, 1=Cesariana")
    
    # 11. h05_chupeta_usou - variable binária (usou de alguma forma vs não usou)
    # Considerar como "usou": "Já usou chupeta, mas não usa mais" + "Usa chupeta"
    # Considerar como "não usou": "Recusou", "Nunca usou", "Não sabe"
    chupeta_usou = df['h05_chupeta_usou'].isin(['Já usou chupeta, mas não usa mais', 'Usa chupeta'])
    df_fe['chupeta_usou'] = chupeta_usou.astype(int)
    
    print("[OK] h05_chupeta_usou: converted to chupeta_usou (0=não usou, 1=usou)")
    print(f"   'Used' includes: 'Já usou' + 'Usa chupeta' = {chupeta_usou.sum()} cases")
    print(f"   'Not used' includes: 'Recusou' + 'Nunca oferecido' + 'Não sabe' = {(~chupeta_usou).sum()} cases")
    
    # Add all other columns that were not processed
    processed_vars = [
        'id_anon', 'a00_regiao', 'b02_sexo', 'b04_idade', 'bb04_idade_da_mae', 
        'd01_cor', 'h01_semanas_gravidez', 'h02_peso', 'h03_altura', 
        'h04_parto', 'h05_chupeta_usou'
    ]
    
    remaining_vars = [col for col in df.columns if col not in processed_vars]
    
    # Add remaining columns to the final dataset
    for col in remaining_vars:
        df_fe[col] = df[col]
    
    print(f"\n[OK] Added {len(remaining_vars)} remaining variables without modification:")
    for i, col in enumerate(remaining_vars, 1):
        if i <= 10:  # Mostrar apenas as primeiras 10
            print(f"   {i:2d}. {col}")
        elif i == 11:
            print(f"   ... and {len(remaining_vars)-10} variables")
            break
    
    # Save result
    output_dir = '/Users/marcelosilva/Desktop/early-obesity-prediction/C-Feature Engeneering'
    os.makedirs(output_dir, exist_ok=True)
    
    output_path = os.path.join(output_dir, '1FE1-10.csv')
    df_fe.to_csv(output_path, index=False)
    
    print("\n" + "=" * 60)
    print("FEATURE ENGINEERING COMPLETED!")
    print("=" * 60)
    print(f"Original dataset: {df.shape}")
    print(f"Final dataset: {df_fe.shape}")
    print(f"Arquivo salvo: {output_path}")
    print(f"Original variables processed: 11")
    print(f"Transformed variables created: {21}")
    print(f"Remaining variables kept: {len(remaining_vars)}")
    print(f"Total variables in final dataset: {df_fe.shape[1]}")
    
    # Resumo das colunas createds
    print("\nColumns in final dataset:")
    for i, col in enumerate(df_fe.columns, 1):
        print(f"{i:2d}. {col}")
    
    return df_fe

# Execute feature engineering
if __name__ == "__main__":
    df_result = feature_engineering_1_11()
    
    # Verificações finais
    print("\n" + "=" * 60)
    print("FINAL CHECKS:")
    print("=" * 60)
    
    # Missing values
    missing_summary = df_result.isnull().sum()
    if missing_summary.sum() > 0:
        print("[WARNING]  Missing values found:")
        print(missing_summary[missing_summary > 0])
    else:
        print("[OK] No missing values in final dataset")
    
    # Tipos de dados
    print(f"\nData types:")
    print(df_result.dtypes.value_counts())

Original dataset: (8236, 56)
Starting Feature Engineering of the first 11 variables...
[OK] id_anon: kept as identifier
[OK] a00_regiao: converted to 5 dummy variables
   Regions: ['regiao_Centro-Oeste', 'regiao_Nordeste', 'regiao_Norte', 'regiao_Sudeste', 'regiao_Sul']
[OK] b02_sexo: converted to sexo_masculino (0=Feminino, 1=Masculino)
[OK] b04_idade: kept as idade_crianca
[OK] bb04_idade_da_mae: created idade_materna_ao_nascimento (bb04 - b04)
   Missing values filled with median: 29.0
[OK] d01_cor: small categories grouped into 'Outros' e converted to matriz
   Categories: ['cor_Branca', 'cor_Outros', 'cor_Parda (mulata, cabocla, cafuza, mameluca ou mestiça)', 'cor_Preta']
[OK] h01_semanas_gravidez: kept as semanas_gravidez
[OK] h02_peso: kept as peso_nascimento
[OK] h03_altura: kept as altura_nascimento
[OK] h04_parto: converted to matrix + binary variable tipo_de_parto
   Matrix categories: ['parto_Cesariana agendada (eletiva)', 'parto_Cesariana de urgência (Não agendada)', 'part